In [0]:
%pip install yfinance pandas
%pip install requests

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 119.6 MB/s eta 0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15548 sha256=95e182027e86c7e2bfc1cb3007767899a851615a38c32e65922b21fe55195468
  Stored in directory: /home/spark-78c09002-f4f8-45df-a2a1-fb/.cache/pip/wheels/cc/bd/6f/664d62c99327abeef7d86489e6631cbf45b56fbf7ef1d6ef00
Successfully built multitasking
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import requests
import time
import pytz
from datetime import datetime
from pyspark.sql import Row

SYMBOL_CB = "ETH-USD"
API_URL = f"https://api.exchange.coinbase.com/products/{SYMBOL_CB}/ticker"
SLEEP_TIME = 5 
TABLE_NAME = "eth_raw_stream_table" 
PL_TIMEZONE = pytz.timezone('Europe/Warsaw')

print(f"producent danych uruchomiony (Czas PL). Dane trafiają do tabeli '{TABLE_NAME}'.")

while True:
    try:
        response = requests.get(API_URL, timeout=10)
        response.raise_for_status() 
        data_json = response.json()

        #plzone
        current_time = datetime.now(PL_TIMEZONE)
        timestamp_clean = current_time.replace(tzinfo=None)
        price = float(data_json.get('price', 0.0))
        volume = float(data_json.get('volume_24h', 0.0))

        if price > 0.0:
            record = {
                "timestamp": timestamp_clean,
                "open": price, "close": price,
                "high": price, "low": price,
                "volume": volume
            }

            # Zapis do tabeli bo darmowy databrick blokuje doslownie wsztstko
            df_spark = spark.createDataFrame([Row(**record)])
            
            (df_spark.write
                     .format("delta")
                     .mode("append")
                     .saveAsTable(TABLE_NAME))

            print(f"[{timestamp_clean.strftime('%H:%M:%S')}] Zapisano: {price:.2f}")
        
    except Exception as e:
        print(f"Błąd: {e}")

    time.sleep(SLEEP_TIME)

Producent danych uruchomiony (Czas PL). Dane trafiają do tabeli 'eth_raw_stream_table'.
[21:38:23] Zapisano: 3104.95
[21:38:30] Zapisano: 3103.87
[21:38:37] Zapisano: 3103.75
[21:38:46] Zapisano: 3103.75
[21:38:53] Zapisano: 3103.75
[21:39:00] Zapisano: 3103.75
[21:39:06] Zapisano: 3103.40
[21:39:13] Zapisano: 3103.66
[21:39:20] Zapisano: 3103.60
[21:39:26] Zapisano: 3104.49
[21:39:33] Zapisano: 3104.48
[21:39:39] Zapisano: 3104.80
[21:39:46] Zapisano: 3104.81
[21:39:53] Zapisano: 3105.00
[21:39:59] Zapisano: 3104.51
[21:40:06] Zapisano: 3104.51
[21:40:13] Zapisano: 3103.73
[21:40:19] Zapisano: 3104.10
[21:40:26] Zapisano: 3104.63
[21:40:32] Zapisano: 3105.18
[21:40:39] Zapisano: 3105.17
[21:40:45] Zapisano: 3105.54
[21:40:52] Zapisano: 3102.59
[21:40:59] Zapisano: 3101.90
[21:41:05] Zapisano: 3100.97
[21:41:12] Zapisano: 3101.18
[21:41:19] Zapisano: 3101.29
[21:41:27] Zapisano: 3101.40
[21:41:33] Zapisano: 3100.69
[21:41:40] Zapisano: 3100.60
[21:41:46] Zapisano: 3099.86
[21:41:52] Za

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can